# Channel Semantic-Space Performance Analysis

This notebook measures how video engagement changes as a function of **semantic distance from a channel-specific engagement optimum** in the original **20D video embedding space**.  

It replaces cluster-only evaluation with a continuous-space analysis that answers four goals:

1. Find the point where engagement is optimal per channel.
2. Measure correlation between semantic distance and engagement.
3. Rank channels by engagement predictability in semantic space.
4. Produce plots that make the relationships interpretable.


## 1) Setup, paths, and reproducibility controls

This cell imports analysis libraries, sets deterministic behavior, and defines the canonical input artifact path for 20D video embeddings.


In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import euclidean_distances
from scipy.stats import spearmanr, pearsonr
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt

# Optional for Colab workflows
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/Graphiko/analysis/channel_semantic_performance')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_VIDEOS_PER_CHANNEL = 15
MIN_STD_LOG_VIEWS = 1e-9


## 2) Load data and enforce schema contract

This step validates the dataset and confirms we are reading a 20D embedding artifact (not a 2D projection-only file). It also normalizes labels to keep `channel_name` as the primary human-readable identifier.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'20D embeddings artifact not found at: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)

required = {'channel_name', 'video_id', 'video_title', 'view_count'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

# Prefer the canonical name label for all outputs.
if 'channel_id' not in df.columns:
    df['channel_id'] = None

# Detect embedding representation.
embedding_col = None
dim_cols = [c for c in df.columns if c.startswith('embedding_20d_')]
if 'embedding_20d' in df.columns:
    embedding_col = 'embedding_20d'
elif len(dim_cols) == 20:
    dim_cols = sorted(dim_cols)
else:
    raise ValueError(
        'Expected either `embedding_20d` (list-like) or exactly 20 scalar columns named `embedding_20d_*`.'
    )

print(f'Loaded rows: {len(df):,}')
print(f'Unique channels: {df["channel_name"].nunique():,}')
print('Embedding schema:', 'embedding_20d column' if embedding_col else '20 scalar embedding columns')


## 3) Parse 20D embeddings and build clean analysis frame

This cell converts embeddings into numeric arrays, computes log-engagement, and filters unusable rows. The resulting frame is the consistent base for all downstream calculations.


In [ ]:
def parse_embedding(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        arr = np.asarray(x, dtype=float)
    elif isinstance(x, str):
        arr = np.asarray(ast.literal_eval(x), dtype=float)
    else:
        return None
    return arr if arr.shape == (20,) else None

if embedding_col:
    df['embedding_vec'] = df[embedding_col].apply(parse_embedding)
else:
    df['embedding_vec'] = df[dim_cols].to_numpy(dtype=float).tolist()

work = df[df['embedding_vec'].notna()].copy()
work['view_count'] = pd.to_numeric(work['view_count'], errors='coerce')
work = work[work['view_count'].notna() & (work['view_count'] >= 0)].copy()
work['log_views'] = np.log1p(work['view_count'])

print(f'Rows after embedding/view filters: {len(work):,}')


## 4) Estimate channel-specific engagement-optimal semantic points

For each eligible channel, we estimate a weighted centroid in 20D where weights are proportional to centered positive engagement (`log_views`).

Intuition: videos with stronger-than-channel-average engagement pull the optimum toward their semantic neighborhood.


In [ ]:
def weighted_optimum(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    centered = y - y.mean()
    w = np.clip(centered, a_min=0.0, a_max=None)
    if np.allclose(w.sum(), 0.0):
        w = np.ones_like(y)
    w = w / w.sum()
    return (X * w[:, None]).sum(axis=0)

rows = []
channel_level = []

for channel_name, g in work.groupby('channel_name', sort=True):
    g = g.reset_index(drop=True).copy()
    n = len(g)
    y = g['log_views'].to_numpy(dtype=float)

    if n < MIN_VIDEOS_PER_CHANNEL or float(np.std(y)) <= MIN_STD_LOG_VIEWS:
        channel_level.append({
            'channel_name': channel_name,
            'channel_id': g['channel_id'].iloc[0],
            'n_videos': n,
            'eligible': False,
            'note': 'Insufficient rows or no engagement variance'
        })
        continue

    X = np.vstack(g['embedding_vec'].to_numpy())
    opt = weighted_optimum(X, y)
    d = euclidean_distances(X, opt.reshape(1, -1)).ravel()

    rho_s, p_s = spearmanr(d, y)
    rho_p, p_p = pearsonr(d, y)

    X_reg = sm.add_constant(d)
    model = sm.OLS(y, X_reg).fit()

    g['distance_to_optimum'] = d
    g['channel_optimum_20d'] = [opt.tolist()] * len(g)

    rows.append(g)
    channel_level.append({
        'channel_name': channel_name,
        'channel_id': g['channel_id'].iloc[0],
        'n_videos': n,
        'eligible': True,
        'spearman_r': float(rho_s),
        'spearman_p': float(p_s),
        'pearson_r': float(rho_p),
        'pearson_p': float(p_p),
        'linear_slope': float(model.params[1]),
        'linear_r2': float(model.rsquared),
        'linear_adj_r2': float(model.rsquared_adj),
        'optimum_norm_l2': float(np.linalg.norm(opt)),
        'note': ''
    })

video_scored = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
channel_metrics = pd.DataFrame(channel_level)

channel_metrics.head()


## 5) Rank channels by engagement predictability

Predictability is operationalized by monotonic and linear signal strength between distance and engagement. We rank by adjusted R² (primary), then absolute Spearman correlation (secondary).


In [ ]:
eligible = channel_metrics[channel_metrics['eligible']].copy()
if eligible.empty:
    raise RuntimeError('No eligible channels after filtering. Lower MIN_VIDEOS_PER_CHANNEL or inspect input data quality.')

eligible['abs_spearman_r'] = eligible['spearman_r'].abs()
ranking = eligible.sort_values(
    by=['linear_adj_r2', 'abs_spearman_r', 'n_videos'],
    ascending=[False, False, False]
).reset_index(drop=True)

ranking['predictability_rank'] = np.arange(1, len(ranking) + 1)

ranking[['predictability_rank','channel_name','n_videos','linear_adj_r2','spearman_r','linear_slope']].head(20)


## 6) Visual diagnostics

The first plot compares predictability across channels. The second plot shows pooled relationship patterns with channel-level normalization to reduce scale effects.


In [ ]:
sns.set_theme(style='whitegrid')

top_n = min(20, len(ranking))
plot_df = ranking.head(top_n).sort_values('linear_adj_r2', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.35)))
sns.barplot(data=plot_df, x='linear_adj_r2', y='channel_name', palette='viridis', ax=ax)
ax.set_title('Top channels by engagement predictability (Adjusted $R^2$)')
ax.set_xlabel('Adjusted $R^2$: log_views ~ distance_to_optimum')
ax.set_ylabel('Channel')
plt.tight_layout()
plt.show()

z = video_scored.copy()
z['distance_z'] = z.groupby('channel_name')['distance_to_optimum'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))
z['log_views_z'] = z.groupby('channel_name')['log_views'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=z, x='distance_z', y='log_views_z', scatter_kws={'alpha': 0.15, 's': 12}, line_kws={'color': 'red'}, ax=ax)
ax.set_title('Pooled channel-normalized relationship: distance vs engagement')
ax.set_xlabel('Distance to channel optimum (z-score within channel)')
ax.set_ylabel('log_views (z-score within channel)')
plt.tight_layout()
plt.show()


## 7) Export artifacts for downstream use

We export both channel-level metrics (for ranking/reporting) and video-level scored rows (for drill-down and interactive visualization).


In [ ]:
channel_metrics_path = OUTPUT_DIR / 'channel_semantic_predictability_metrics.csv'
ranking_path = OUTPUT_DIR / 'channel_predictability_ranking.csv'
video_scored_path = OUTPUT_DIR / 'video_semantic_distance_scored.csv'

channel_metrics.to_csv(channel_metrics_path, index=False)
ranking.to_csv(ranking_path, index=False)
video_scored.to_csv(video_scored_path, index=False)

print('Wrote:')
print('-', channel_metrics_path)
print('-', ranking_path)
print('-', video_scored_path)
